## Calcular quantidade de beneficiários do bolsa família e abandono por estado

In [1]:
import pandas as pd

In [ ]:
uf_map = {
    11: {'sigla': 'RO', 'nome': 'Rondônia'},
    12: {'sigla': 'AC', 'nome': 'Acre'},
    13: {'sigla': 'AM', 'nome': 'Amazonas'},
    14: {'sigla': 'RR', 'nome': 'Roraima'},
    15: {'sigla': 'PA', 'nome': 'Pará'},
    16: {'sigla': 'AP', 'nome': 'Amapá'},
    17: {'sigla': 'TO', 'nome': 'Tocantins'},
    21: {'sigla': 'MA', 'nome': 'Maranhão'},
    22: {'sigla': 'PI', 'nome': 'Piauí'},
    23: {'sigla': 'CE', 'nome': 'Ceará'},
    24: {'sigla': 'RN', 'nome': 'Rio Grande do Norte'},
    25: {'sigla': 'PB', 'nome': 'Paraíba'},
    26: {'sigla': 'PE', 'nome': 'Pernambuco'},
    27: {'sigla': 'AL', 'nome': 'Alagoas'},
    28: {'sigla': 'SE', 'nome': 'Sergipe'},
    29: {'sigla': 'BA', 'nome': 'Bahia'},
    31: {'sigla': 'MG', 'nome': 'Minas Gerais'},
    32: {'sigla': 'ES', 'nome': 'Espírito Santo'},
    33: {'sigla': 'RJ', 'nome': 'Rio de Janeiro'},
    35: {'sigla': 'SP', 'nome': 'São Paulo'},
    41: {'sigla': 'PR', 'nome': 'Paraná'},
    42: {'sigla': 'SC', 'nome': 'Santa Catarina'},
    43: {'sigla': 'RS', 'nome': 'Rio Grande do Sul'},
    50: {'sigla': 'MS', 'nome': 'Mato Grosso do Sul'},
    51: {'sigla': 'MT', 'nome': 'Mato Grosso'},
    52: {'sigla': 'GO', 'nome': 'Goiás'},
    53: {'sigla': 'DF', 'nome': 'Distrito Federal'}
}

ano = 2015 

df = pd.read_csv("bolsa_familia/dadosLimpos/valorRepassado_familia_2015.csv")

df['cod_uf'] = df['ibge'].astype(str).str[:2].astype(int)
df['uf_sigla'] = df['cod_uf'].map(lambda x: uf_map[x]['sigla'])
df['uf_nome'] = df['cod_uf'].map(lambda x: uf_map[x]['nome'])


df_estado = df.groupby(['uf_sigla', 'uf_nome']).agg({
    'valor_repassado_bolsa_familia': 'sum',
    'qtd_familias_beneficiarias_bolsa_familia': 'sum'
}).reset_index()


df_estado['valor_medio_bf'] = df_estado['valor_repassado_bolsa_familia'] / df_estado['qtd_familias_beneficiarias_bolsa_familia']
df_estado['ano'] = ano 

df_estado = df_estado.sort_values('uf_sigla')
resultado = df_estado[['ano', 'uf_sigla', 'uf_nome', 'qtd_familias_beneficiarias_bolsa_familia', 'valor_repassado_bolsa_familia', 'valor_medio_bf']]

nome_arquivo_saida = f'bolsa_familia_por_estado_{ano}.csv'
resultado.to_csv(nome_arquivo_saida, index=False, float_format='%.2f')

Arquivo salvo: bolsa_familia_por_estado_2023.csv
    ano uf_sigla   uf_nome  qtd_familias_beneficiarias_bolsa_familia_s  \
0  2023       AC      Acre                                     1310025   
1  2023       AL   Alagoas                                     5449440   
2  2023       AM  Amazonas                                     6362348   
3  2023       AP     Amapá                                     1227549   
4  2023       BA     Bahia                                    25393633   

   pbf_vlr_medio_benef_f  valor_medio_estadual  
0              166009.08           7545.867273  
1              705893.44           6920.523922  
2              468606.41           7558.167903  
3              117496.35           7343.521875  
4             2798837.68           6711.840959  


In [ ]:
df = pd.read_csv('bolsa_familia_por_estado_2015.csv')

if 'valor_medio_estadual' in df.columns:
    df['qtd_familias_beneficiarias_bolsa_familia_ajustado'] = (df['qtd_familias_beneficiarias_bolsa_familia_s'] / 12).round(0)

    df.to_csv('bolsa_familia/dadosTratados/bolsa_familia_por_estado_2015.csv', index=False)
    
    print("Ajuste realizado com sucesso! Resultado com 2 casas decimais:")
    print(df[['uf_sigla', 'qtd_familias_beneficiarias_bolsa_familia_s', 'qtd_familias_beneficiarias_bolsa_familia_ajustado']].head())
else:
    print("Erro: Coluna 'valor_medio_bf' não encontrada.")
    print("Colunas disponíveis:", list(df.columns))

Ajuste realizado com sucesso! Resultado com 2 casas decimais:
  uf_sigla  qtd_familias_beneficiarias_bolsa_familia_s  \
0       AC                                     1310025   
1       AL                                     5449440   
2       AM                                     6362348   
3       AP                                     1227549   
4       BA                                    25393633   

   qtd_familias_beneficiarias_bolsa_familia_ajustado  
0                                           109169.0  
1                                           454120.0  
2                                           530196.0  
3                                           102296.0  
4                                          2116136.0  


In [ ]:
df = pd.read_csv('analise_robusta2.csv')

cols_beneficiarias = [col for col in df.columns if 'qtd_familias_beneficiarias_bolsa_familia_ajustado_' in col]
df['Total_Beneficiarios'] = df[cols_beneficiarias].sum(axis=1)
total_geral_ef = df['Total_Beneficiarios'].sum()

colunas_selecionadas = ['Estado', 'UF', 'Total_Beneficiarios']
df_final = df[colunas_selecionadas].copy()
df_final = df_final.sort_values('Total_Beneficiarios', ascending=False)

df_final.to_csv('dados_com_beneficiarios_totais.csv', index=False, encoding='utf-8-sig')


### Quantidade total de abandono escolar por estado (2012-2021)

In [ ]:
df = pd.read_csv('analise_robusta2.csv')


cols_abandono_ef = [col for col in df.columns if 'Abandono_EF' in col]
cols_abandono_em = [col for col in df.columns if 'Abandono_EM' in col]


df['Total_Abandono_EF'] = df[cols_abandono_ef].sum(axis=1)
df['Total_Abandono_EM'] = df[cols_abandono_em].sum(axis=1)


total_geral_ef = df['Total_Abandono_EF'].sum()
total_geral_em = df['Total_Abandono_EM'].sum()


df.to_csv('dados_com_abandonos_totais.csv', index=False, encoding='utf-8-sig')

colunas_selecionadas = ['Estado', 'UF', 'Total_Abandono_EF', 'Total_Abandono_EM']
df_final = df[colunas_selecionadas].copy()
df_final = df_final.sort_values('Total_Abandono_EF', ascending=False)


df_final.to_csv('dados_com_abandonos_totais.csv', index=False, encoding='utf-8-sig')

In [ ]:
df = pd.read_csv('dados_com_abandonos_totais.csv')

df['Total_Abandono_EF'] = df.filter(like='Abandono_EF').sum(axis=1)
df['Total_Abandono_EM'] = df.filter(like='Abandono_EM').sum(axis=1)

df['Total_Geral'] = df['Total_Abandono_EF'] + df['Total_Abandono_EM']
df_final = df[['Estado', 'UF', 'Total_Geral']]


df_final.to_csv('total_abandono_por_estado.csv', index=False, encoding='utf-8-sig')